# Análisis de Sentimiento de Reviews

Proyecto de clasificación de texto listo para **Google Colab**.

**Pregunta:** ¿podemos predecir si una review es positiva o negativa?

**Target:** sentimiento (positivo/negativo)

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset de ejemplo de reviews.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar / simular datos de reviews

In [ ]:
np.random.seed(RANDOM_STATE)
# Reviews simuladas simples
positivas = ["Excelente producto, muy recomendado", "Me encantó, calidad superior", "Muy bueno, lo volvería a comprar", "Perfecto, cumplió expectativas", "Gran calidad y servicio"]
negativas = ["Pésimo producto, no funciona", "Muy malo, no lo recomiendo", "Decepcionado, calidad baja", "No vale la pena", "Mal servicio y producto"]

reviews = []
labels = []
for _ in range(200):
    if np.random.rand() > 0.5:
        reviews.append(np.random.choice(positivas) + " " + np.random.choice(["", "Genial", "5 estrellas"]))
        labels.append(1)
    else:
        reviews.append(np.random.choice(negativas) + " " + np.random.choice(["", "Horrible", "1 estrella"]))
        labels.append(0)

df = pd.DataFrame({"review":: reviews, "sentimiento":: labels})
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df["sentimiento"].value_counts())
print(df["sentimiento"].value_counts(normalize=True).round(3))

In [ ]:
sns.countplot(data=df, x="sentimiento")
plt.title("Distribución de sentimiento")
plt.xticks([0, 1], ["Negativo", "Positivo"])
plt.show()

### Hallazgos
- Dataset balanceado.
- Texto corto ideal para TF-IDF.
- Listo para clasificación de sentimiento.

## 3. Preparar datos

In [ ]:
X = df["review"]
y = df["sentimiento"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos con TF-IDF

In [ ]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=1000)),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)

## 5. Evaluación detallada

In [ ]:
best = pipes["Logistic Regression"]
y_pred = best.predict(X_test)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de confusión")
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.show()

## 6. SHAP (aproximado para texto)

In [ ]:
!pip install -q shap

In [ ]:
import shap

# Usar explainer sobre el modelo lineal
vectorizer = best.named_steps["tfidf"]
model = best.named_steps["model"]

X_train_vec = vectorizer.transform(X_train)
explainer = shap.LinearExplainer(model, X_train_vec)
shap_values = explainer.shap_values(X_train_vec)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values, X_train_vec, feature_names=vectorizer.get_feature_names_out(), max_display=10, show=False)
plt.title("SHAP Summary — Sentimiento")
plt.tight_layout()
plt.show()

## 7. Conclusiones

- TF-IDF + Logistic Regression funciona muy bien en este ejemplo.
- Palabras clave positivas/negativas dominan las predicciones.
- Se puede extender a reviews reales de Amazon, IMDB, etc.

**Rango esperado (ejemplo):** accuracy > 0.90